# Colab → Cursor (`google-colab-qwen2.5-coder-3b`)

Sobe vLLM na GPU + túnel HTTPS e **regista sozinho** no proxy do PC (Funnel Tailscale).

No PC:
```bash
python3 bin/prepare-colab-notebook.py
# abre notebooks/colab_cursor_coder.ready.ipynb → Kernel Colab (GPU) → Run All
```

No picker do Cursor: **google-colab-qwen2.5-coder-3b**. Mantém a sessão Colab ligada.


In [ ]:
!nvidia-smi -L


In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"  # cabe em T4; 7B se VRAM aguentar
PORT = 8001
print(MODEL, PORT)


In [ ]:
!pip -q install -U "vllm>=0.8.0" pyngrok


In [ ]:
import pathlib, subprocess, time

log = pathlib.Path("vllm_colab.log")
subprocess.run(["bash", "-lc", "pkill -f 'vllm serve' || true"], check=False)
cmd = (
    f"nohup vllm serve {MODEL} --host 0.0.0.0 --port {PORT} "
    f"--dtype half --max-model-len 8192 --gpu-memory-utilization 0.92 "
    f"> {log} 2>&1 &"
)
subprocess.run(["bash", "-lc", cmd], check=True)
print("a carregar", MODEL)
ready = False
text = ""
for i in range(180):
    time.sleep(2)
    text = log.read_text(errors="replace") if log.exists() else ""
    if "Application startup complete" in text or "Uvicorn running" in text:
        ready = True
        break
    if i % 15 == 0:
        print(f"… {i*2}s")
print("READY" if ready else "TIMEOUT — vê vllm_colab.log")
print(text[-1000:])


In [ ]:
import json, urllib.request
with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=10) as r:
    print(json.dumps(json.load(r), indent=2)[:1000])


## Túnel HTTPS (cloudflared; fallback ngrok se `NGROK_AUTHTOKEN`)


In [ ]:
import json, os, re, shutil, subprocess, time, urllib.request

subprocess.run(["bash", "-lc", "pkill -f 'cloudflared tunnel' || true"], check=False)
if not shutil.which("cloudflared"):
    subprocess.run([
        "bash", "-lc",
        "curl -fsSL -o /tmp/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "&& chmod +x /tmp/cloudflared"
    ], check=True)
    cloudflared = "/tmp/cloudflared"
else:
    cloudflared = "cloudflared"

logf = "/tmp/cloudflared_colab.log"
subprocess.Popen(
    [cloudflared, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open(logf, "w"),
    stderr=subprocess.STDOUT,
)
public = None
for _ in range(50):
    time.sleep(0.4)
    try:
        text = open(logf, errors="replace").read()
    except OSError:
        text = ""
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if m:
        public = m.group(0)
        break

if not public:
    token = os.environ.get("NGROK_AUTHTOKEN", "")
    if not token:
        raise RuntimeError("sem túnel cloudflared e sem NGROK_AUTHTOKEN")
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    public = ngrok.connect(PORT, "http").public_url

public = public.rstrip("/")
print("COLAB_PUBLIC_URL=", public)

req = urllib.request.Request(
    public + "/v1/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say hi in one word."}],
        "max_tokens": 8,
    }).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer sk-colab"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=180) as r:
    out = json.load(r)
print("smoke:", out["choices"][0]["message"]["content"])

# Auto-registo no proxy Cursor (Funnel). prepare-colab-notebook.py preenche REGISTER_*.
REGISTER_BASE = globals().get("REGISTER_BASE", "")
REGISTER_TOKEN = globals().get("REGISTER_TOKEN", "")
if REGISTER_BASE and REGISTER_TOKEN:
    reg = REGISTER_BASE.rstrip("/") + "/v1/colab/register"
    rreq = urllib.request.Request(
        reg,
        data=json.dumps({"url": public, "model": MODEL}).encode(),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {REGISTER_TOKEN}",
        },
        method="POST",
    )
    with urllib.request.urlopen(rreq, timeout=60) as rr:
        rout = json.load(rr)
    print("REGISTERED:", rout)
    print("picker →", rout.get("picker") or "google-colab-qwen2.5-coder-3b")
    print("Fecha o Cursor uns segundos (BYOK) ou: ./install.sh cursor --yes")
else:
    print("REGISTER_BASE/TOKEN ausentes — no PC: python3 bin/prepare-colab-notebook.py")
    print(f"./install.sh cursor --colab-url '{public}' --colab-model '{MODEL}' --yes")


Mantém a sessão Colab **ligada**. Se a VM cair, Run All de novo (o registo atualiza o túnel sozinho).
